# Any-Precision: CNN-8 op SVHN, alleen train
## Volledige no-MX-referentie: 100 epochs vanaf initialisatie

Dit notebook **leest uitsluitend opgeslagen onderzoeksgegevens**. Het traint niet, downloadt niets en verandert geen checkpoints. Alle 73.257 beelden van de originele `train`-split worden gebruikt; `extra` wordt niet gebruikt. Er is geen afzonderlijke validatiesplit. Een afgebroken run wordt hieronder niet als voltooide honderd-epochreferentie voorgesteld.

Bron: *Any-Precision Deep Neural Networks*, aangeleverde PDF, p. 5, Implementation Details; p. 6 (gedrukte p. 10768), tabel 1, CNN-8, kolommen **Ours**. De paper gebruikt `train + extra`. Deze train-only-variant heeft daarom een ander data- en updatebudget; de literatuurwaarden zijn context, geen gelijkwaardige reproductiedrempel. Architectuur, kwantisator, loss en trainingsfunctie komen uit het aangeleverde auteursarchief. Er is geen MX of truncatie uit opgeslagen codes.

In [ ]:
from pathlib import Path
RUN_DIR = Path(".")

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from IPython.display import display, Markdown

def read_json(name):
    return json.loads((RUN_DIR / name).read_text(encoding="utf-8"))

required = ["run_report.json", "epoch_metrics.csv", "selected_metrics.csv",
            "checkpoint_selection.json", "epoch_times.csv", "timing_estimate.json",
            "completion_summary.json", "learning_rate_schedule.csv", "history.json"]
missing = [name for name in required if not (RUN_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Ontbrekende resultaten in {RUN_DIR}: {missing}. Open het notebook uit de runmap, na afloop.")
report = read_json("run_report.json")
metrics = pd.read_csv(RUN_DIR / "epoch_metrics.csv")
selected = pd.read_csv(RUN_DIR / "selected_metrics.csv")
times = pd.read_csv(RUN_DIR / "epoch_times.csv")
lrs = pd.read_csv(RUN_DIR / "learning_rate_schedule.csv")
selection = read_json("checkpoint_selection.json")
estimate = read_json("timing_estimate.json")
completion = read_json("completion_summary.json")
history = read_json("history.json")
if report.get("synthetic_fixture_only"):
    display(Markdown("**SYNTHETISCHE CPU-SOFTWARETEST. Geen SVHN-resultaten en geen bruikbare GPU-tijden.**"))
display(Markdown(f"**Status:** {report['status']} | **Afgerond:** {completion['completed_epochs']}/100 epochs | "
                 f"**Volledige training:** {completion['full_training_completed']}"))
if not completion["full_training_completed"]:
    display(Markdown("**ONVOLLEDIG: alleen resultaten van de afgeronde epochs. Dit is nog geen honderd-epochreferentie.**"))
display(pd.DataFrame([report.get("environment", {})]).T.rename(columns={0: "waarde"}))

## 1. Protocol en integriteitscontroles

CNN-8 (`svhnq`, `expand=2`), Resize(40), batch 128, Adam zonder weight decay, initiele LR 0,001. Vijf precisies met recursieve distillatie: 32 -> 8 -> 4 -> 2 -> 1. Dropout, per-precisie BatchNorm en de eerste/laatste laag blijven zoals geimplementeerd. Evaluatie op de volledige testset na iedere epoch; `val` in de auteurscode verwijst naar `test`.

Alle 100 epochs zijn vooraf gepland. LR-verlaging na epochs 50/75/90. Dat zijn 57.300 optimizerstappen bij train-only, tegenover 472.200 voor het train+extra-recept. Geen prestatiegestuurd vroegtijdig stoppen. Een noodgeval kan de uitvoering wel onderbreken.

In [ ]:
cfg = report["config"]
display(pd.DataFrame(list(cfg.items()), columns=["instelling", "waarde"]))
display(pd.DataFrame(list(report.get("execution_settings", {}).items()), columns=["uitvoering", "waarde"]))
assert cfg["train_split"] == "train" and cfg["extra_used"] is False
assert cfg["planned_epochs"] == cfg["target_final_epoch"] == 100
assert cfg["scheduler_milestones"] == [50, 75, 90]
assert not metrics.duplicated(["epoch", "phase", "bits_w_and_a"]).any()
assert np.isfinite(metrics[["objective_loss", "top1_percent", "top5_percent", "lr"]].to_numpy()).all()
assert len(metrics) == 10 * completion["completed_epochs"]
assert sorted(metrics.epoch.unique()) == list(range(1, completion["completed_epochs"] + 1))
for (_, phase), rows in metrics.groupby(["epoch", "phase"]):
    assert sorted(rows.bits_w_and_a) == [1, 2, 4, 8, 32]
assert completion["lr_schedule_verified"] is True
if not report.get("synthetic_fixture_only"):
    assert cfg["train_examples"] == 73257 and cfg["test_examples"] == 26032
    assert set(report["dataset_files"]["files"]) == {"train_32x32.mat", "test_32x32.mat"}
    assert report["dataset_selection"]["train_split_order"] == ["train"]
    assert set(metrics.loc[metrics.phase == "train", "n"]) == {73257}
    assert set(metrics.loc[metrics.phase == "test", "n"]) == {26032}
    assert completion["optimizer_steps"] == 573 * completion["completed_epochs"]
    display(pd.DataFrame([{"split": name, **report["dataset_selection"][name]} for name in ["train", "test"]]))
display(Markdown("**De bovenstaande controles zijn geslaagd.** Ze controleren opgeslagen rapporten; het notebook laadt de modelgewichten niet opnieuw."))

## 2. Leercurves

Trainingsaccuracy is gemeten met dropout actief tijdens de updates. Testaccuracy is gemeten na de epoch in evaluatiemodus. Daarom zijn deze curves niet onder identieke omstandigheden berekend. Er wordt geen gewenste rangorde van precisies afgedwongen.

In [ ]:
for phase, title in [("train", "Trainingsaccuracy"), ("test", "Testaccuracy")]:
    fig, ax = plt.subplots(figsize=(10, 4.5))
    for bit in [1, 2, 4, 8, 32]:
        rows = metrics[(metrics.phase == phase) & (metrics.bits_w_and_a == bit)].sort_values("epoch")
        ax.plot(rows.epoch, rows.top1_percent, label=f"W{bit}/A{bit}")
    ax.set(title=title, xlabel="Epoch", ylabel="Top-1-accuracy (%)")
    ax.xaxis.set_major_locator(MaxNLocator(nbins=10, integer=True))
    ax.grid(True, alpha=0.25)
    ax.legend(ncol=3)
    fig.tight_layout()
    plt.show()

## 3. Trainingsdoel en learning rate

De hoogste precisie gebruikt hard-label cross-entropy. De lagere precisies gebruiken tijdens training recursieve soft-target cross-entropy, **geen afzonderlijke hard-label CE en geen rechtstreekse KL-meting**. De testloss gebruikt wel hard-label CE bij alle precisies. Losswaarden worden daarom niet zonder onderscheid als identieke trainingsdoelen vergeleken.

In [ ]:
for phase, title in [("train", "Trainingsloss volgens het gebruikte doel"), ("test", "Testloss: hard-label cross-entropy")]:
    fig, ax = plt.subplots(figsize=(10, 4.5))
    for bit in [1, 2, 4, 8, 32]:
        rows = metrics[(metrics.phase == phase) & (metrics.bits_w_and_a == bit)].sort_values("epoch")
        ax.plot(rows.epoch, rows.objective_loss, label=f"W{bit}/A{bit}")
    ax.set(title=title, xlabel="Epoch", ylabel="Loss")
    ax.xaxis.set_major_locator(MaxNLocator(nbins=10, integer=True))
    ax.grid(True, alpha=0.25)
    ax.legend(ncol=3)
    fig.tight_layout()
    plt.show()

def expected_lr(ep):
    value = cfg["lr"]
    for milestone in cfg["scheduler_milestones"]:
        if milestone < ep:
            value *= cfg["scheduler_gamma"]
    return value
assert np.allclose(lrs.lr_used, [expected_lr(int(ep)) for ep in lrs.epoch], rtol=1e-12, atol=0)
assert np.allclose(metrics.lr, [expected_lr(int(ep)) for ep in metrics.epoch], rtol=1e-12, atol=0)
display(lrs[lrs.epoch.isin([1, 49, 50, 51, 74, 75, 76, 89, 90, 91, 100])])
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.step(lrs.epoch, lrs.lr_used, where="post")
ax.set(xlabel="Epoch", ylabel="Gebruikte learning rate", yscale="log", title="Learning rate per epoch")
ax.xaxis.set_major_locator(MaxNLocator(nbins=10, integer=True))
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()

## 4. Beste checkpoint en laatste epoch

Het beste checkpoint is het eerste strikte maximum van de W32/A32-testaccuracy. Alle vijf precisies worden van **datzelfde checkpoint** gerapporteerd; we kiezen niet voor iedere precisie een eigen beste epoch. Het laatste checkpoint heeft een afzonderlijke rol, ook wanneer de beste epoch toevallig epoch 100 is. Bij onderbreking is 'last' slechts de laatste afgeronde epoch.

De beste testscore is niet onafhankelijk van de checkpointselectie. Dit is de afgesproken auteurscoderegel, geen nieuwe onafhankelijke validatieset.

In [ ]:
display(pd.DataFrame([{"rol": "beste op W32/A32-test", **selection["best"]},
                      {"rol": "laatste afgeronde epoch", **selection["last"]}]))
assert len(selected) == 10
w32 = metrics[(metrics.phase == "test") & (metrics.bits_w_and_a == 32)].sort_values("epoch")
first_best = int(w32.loc[w32.top1_percent.idxmax(), "epoch"])
assert first_best == selection["best"]["epoch"]
assert selection["last"]["epoch"] == completion["completed_epochs"]
comparison = selected.pivot(index=["role", "epoch"], columns="bits_w_and_a", values="top1_percent")
comparison.columns = [f"W{b}/A{b} (%)" for b in comparison.columns]
display(comparison.round(4))
paper = read_json("paper_targets.json")
paper_row = pd.DataFrame([paper["accuracy_percent_by_bits"]], index=["Literatuurcontext: CNN-8, train+extra, 100 epochs, Ours"])
paper_row.columns = [f"W{b}/A{b} (%)" for b in paper_row.columns]
display(paper_row)
display(Markdown("**Geen exacte paperreproductie:** de paper gebruikt train+extra, onze run alleen train. Minder data en optimizerupdates; geen equivalent trainingsbudget."))

## 5. Werkelijke looptijd en extrapolatie

De **gemeten som van de afgewerkte epochs** is het hoofdresultaat voor de epochtijd. Bij 100 afgeronde epochs is dit geen raming meer. Deze som omvat training, testevaluatie, controles en checkpointopslag, maar niet de opstart, datasetintegriteitscontrole/inladen en afsluitende rapportexport. De totale sessieduur staat afzonderlijk. Na hervatten beslaat die laatste alleen het huidige proces.

De oorspronkelijke tijdsraming blijft beschikbaar als diagnostiek, maar wordt niet verward met de werkelijk gemeten honderd-epochduur. Een projectie op basis van minimum/maximum is geen statistisch betrouwbaarheidsinterval. Geen voorspelling voor toekomstige MX-implementaties.

In [ ]:
display(pd.DataFrame([{
    "afgeronde_epochs": completion["completed_epochs"],
    "werkelijk_gemeten_epochsom_min": times.total_epoch_seconds.sum() / 60,
    "gemiddelde_epochtijd_s": times.total_epoch_seconds.mean(),
    "mediane_epochtijd_s": times.total_epoch_seconds.median(),
    "laatste_sessieduur_min": report.get("session_wall_seconds", float("nan")) / 60,
    "training_volledig": completion["full_training_completed"],
}]))
if completion["timing_complete"]:
    assert np.isclose(times.total_epoch_seconds.sum(), completion["measured_completed_epoch_seconds"])
else:
    display(Markdown("**Timing onvolledig:** de niet-gemeten waarden worden niet als nul behandeld."))
fig, ax = plt.subplots(figsize=(10, 4.5))
for name, label in [("train_seconds", "Training"), ("test_seconds", "Test"), ("total_epoch_seconds", "Totaal")]:
    ax.plot(times.epoch, times[name], label=label)
ax.set(xlabel="Epoch", ylabel="Seconden", title="Werkelijk gemeten epochtijden")
ax.xaxis.set_major_locator(MaxNLocator(nbins=10, integer=True))
ax.legend()
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()
if estimate.get("available"):
    display(pd.DataFrame([{"diagnostische_extrapolatie_100_epochs_min": estimate["estimated_100_epoch_seconds"] / 60,
                           "geen_vervanging_voor_gemeten_epochsom": True}]))
if "dataset_load_seconds" in report:
    display(pd.DataFrame([{
        "data_integriteitscontrole_en_eventuele_download_s": report["data_preparation_seconds_including_download_and_checksums"],
        "dataset_inladen_s": report["dataset_load_seconds"],
    }]))
display(times[["epoch", "train_seconds", "test_seconds", "checkpoint_seconds", "other_seconds", "total_epoch_seconds"]].tail(10).round(3))

## 6. Geheugen, waarschuwingen en beperkingen

GPU-geheugen komt uit de PyTorch-registratie en is geen energie- of hardwareprestatiemeting. De auteursinstelling `cudnn.benchmark=True` blijft actief. Een identieke seed en een volledig bewaarde hervatstaat beloven daardoor geen bitexact identieke CUDA-herhaling. `compatibility.patch` beschrijft uitsluitend de bestaande aanpassingen voor uitvoering; `fullrun_change_manifest.json` beschrijft de overgang van de proef naar deze referentie.

In [ ]:
memory = times[["epoch", "peak_gpu_allocated_bytes", "peak_gpu_reserved_bytes"]].copy()
for col in ["peak_gpu_allocated_bytes", "peak_gpu_reserved_bytes"]:
    memory[col.replace("_bytes", "_GiB")] = pd.to_numeric(memory.pop(col), errors="coerce") / 1024**3
display(memory.describe())
checks = {key: report.get(key, "niet gerapporteerd") for key in
          ["source_unchanged", "dataset_files_unchanged", "optimizer_steps", "completed_epoch", "full_training_completed"]}
display(pd.DataFrame(list(checks.items()), columns=["controle", "waarde"]))
warning_rows = report.get("warnings", [])
if warning_rows:
    display(pd.DataFrame(warning_rows))
else:
    display(Markdown("Geen waarschuwingen gerapporteerd in deze sessie."))
for limitation in report.get("limitations", []):
    print(limitation)

## 7. Interpretatie vastleggen

Beschrijf pas na de run de eindresultaten en leercurves. Houd eigen waarnemingen, vergelijking met literatuur en mogelijke verklaringen uit elkaar. Een enkele seed bewijst geen statistisch robuust methodeverschil. Deze run is de train-only-no-MX-referentie, niet het MX-experiment zelf.

Deel `feedback_bundle.zip` uit deze resultatenmap voor analyse. Bewaar lokaal alle ruwe gegevens, checkpoints, bronhashes en dit notebook. De kleine feedbackbundel bevat geen modelgewichten, datasetbeelden of volledige minibatchlogs; de belangrijkste eindcontroles verifieerden die koppelingen tijdens het exporteren.